In [1]:
# imports
import os, glob, math
from collections import deque
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.models import load_model
plt.rcParams['figure.figsize'] = (12,5)
print("TF", tf.__version__)


TF 2.20.0


In [2]:
# Data is in Drive, mount it.
from google.colab import drive
drive.mount('/content/drive')

# Set base folder where your dataset is.
BASE_DIR = '/content/drive/MyDrive/major_data'
os.makedirs(BASE_DIR, exist_ok=True)
print("BASE_DIR:", BASE_DIR)

Mounted at /content/drive
BASE_DIR: /content/drive/MyDrive/major_data


In [ ]:
# Comparison of 3 LSTM models

In [3]:
import pandas as pd
from IPython.display import display

# --- File paths (NOT folders now) ---
M1_PATH = "/content/drive/MyDrive/major_data/LSTM_1/lstm1_metrics/A_lstm1_test_metrics.csv"
M2_PATH = "/content/drive/MyDrive/major_data/LSTM_2/lstm2_metrics/A_lstm2_test_metrics.csv"
M3_PATH = "/content/drive/MyDrive/major_data/LSTM_3/lstm3_metrics/A_lstm3_test_metrics.csv"

# --- Load directly ---
df_m1 = pd.read_csv(M1_PATH)
df_m2 = pd.read_csv(M2_PATH)
df_m3 = pd.read_csv(M3_PATH)

# --- Rename columns ---
df_m1 = df_m1.rename(columns={'RMSE': 'RMSE_M1', 'MAE': 'MAE_M1'})
df_m2 = df_m2.rename(columns={'RMSE': 'RMSE_M2', 'MAE': 'MAE_M2'})
df_m3 = df_m3.rename(columns={'RMSE': 'RMSE_M3', 'MAE': 'MAE_M3'})

# --- Merge ---
comparison_df = pd.merge(df_m1, df_m2, on='Company')
comparison_df = pd.merge(comparison_df, df_m3, on='Company')

# --- Best Model ---
comparison_df['Best_Model'] = comparison_df[
    ['RMSE_M1', 'RMSE_M2', 'RMSE_M3']
].idxmin(axis=1)

# --- Best RMSE ---
comparison_df['Best_RMSE'] = comparison_df[
    ['RMSE_M1', 'RMSE_M2', 'RMSE_M3']
].min(axis=1)

# --- Sort ---
comparison_df = comparison_df.sort_values(by='Best_RMSE').reset_index(drop=True)

# --- Round ---
comparison_df = comparison_df.round(2)

# --- Display ---
styled_table = comparison_df.style.set_properties(**{
    'text-align': 'center'
}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'center')]}
])

display(styled_table)

,Company,RMSE_M1,MAE_M1,RMSE_M2,MAE_M2,RMSE_M3,MAE_M3,Best_Model,Best_RMSE
0,Oil and Natural Gas Corporation Ltd,8.290000,6.480000,9.640000,7.460000,6.210000,4.770000,RMSE_M3,6.210000
1,Larsen and Toubro Ltd,14.220000,10.500000,12.720000,9.760000,10.900000,9.200000,RMSE_M3,10.900000
2,ITC Ltd,11.500000,8.900000,20.360000,15.440000,16.220000,12.080000,RMSE_M1,11.500000
3,JSW Steel Ltd,17.970000,14.740000,12.840000,9.780000,12.320000,9.520000,RMSE_M3,12.320000
4,Bharat Electronics Ltd,14.960000,11.970000,12.620000,10.340000,15.000000,12.580000,RMSE_M2,12.620000
5,Adani ports and special Economic,17.610000,14.780000,15.160000,12.290000,23.100000,19.820000,RMSE_M2,15.160000
6,Power Grid Corporation of India Ltd,28.330000,11.870000,18.570000,7.080000,22.190000,10.880000,RMSE_M2,18.570000
7,Eternal Ltd,23.650000,17.670000,32.380000,24.710000,26.810000,20.230000,RMSE_M1,23.650000
8,Infosys Ltd,49.450000,38.690000,31.930000,23.780000,28.430000,22.060000,RMSE_M3,28.430000
9,Hindustan Unilever Ltd,58.640000,47.450000,43.670000,33.200000,31.300000,23.330000,RMSE_M3,31.300000


In [4]:
from IPython.display import display

# Select required columns
final_table = comparison_df[['Company', 'RMSE_M1', 'RMSE_M2', 'RMSE_M3', 'Best_Model']]

# Round values
final_table = final_table.round(2)

# Styled display
styled_table = final_table.style.set_properties(**{
    'text-align': 'center'
}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'center')]}
])

display(styled_table)

,Company,RMSE_M1,RMSE_M2,RMSE_M3,Best_Model
0,Oil and Natural Gas Corporation Ltd,8.290000,9.640000,6.210000,RMSE_M3
1,Larsen and Toubro Ltd,14.220000,12.720000,10.900000,RMSE_M3
2,ITC Ltd,11.500000,20.360000,16.220000,RMSE_M1
3,JSW Steel Ltd,17.970000,12.840000,12.320000,RMSE_M3
4,Bharat Electronics Ltd,14.960000,12.620000,15.000000,RMSE_M2
5,Adani ports and special Economic,17.610000,15.160000,23.100000,RMSE_M2
6,Power Grid Corporation of India Ltd,28.330000,18.570000,22.190000,RMSE_M2
7,Eternal Ltd,23.650000,32.380000,26.810000,RMSE_M1
8,Infosys Ltd,49.450000,31.930000,28.430000,RMSE_M3
9,Hindustan Unilever Ltd,58.640000,43.670000,31.300000,RMSE_M3


In [5]:
import os

# --- Folder to save comparison CSVs ---
COMPARE_DIR = "/content/drive/MyDrive/major_data/Compared_LSTM"
os.makedirs(COMPARE_DIR, exist_ok=True)

# ---  Save full comparison table ---
full_comparison_path = os.path.join(COMPARE_DIR, "lstm_full_comparison.csv")
comparison_df.to_csv(full_comparison_path, index=False)
print(f"Full comparison saved at: {full_comparison_path}")

# --- Save table with best RMSE per company ---
best_rmse_df = comparison_df[['Company', 'RMSE_M1', 'RMSE_M2', 'RMSE_M3', 'Best_Model']]
best_comparison_path = os.path.join(COMPARE_DIR, "lstm_best_model_rmse.csv")
best_rmse_df.to_csv(best_comparison_path, index=False)
print(f"Best RMSE per company saved at: {best_comparison_path}")


Full comparison saved at: /content/drive/MyDrive/major_data/Compared_LSTM/lstm_full_comparison.csv
Best RMSE per company saved at: /content/drive/MyDrive/major_data/Compared_LSTM/lstm_best_model_rmse.csv


In [6]:
from IPython.display import display
import pandas as pd
import os

# --- Calculate average RMSE and MAE ---
avg_rmse = comparison_df[['RMSE_M1', 'RMSE_M2', 'RMSE_M3']].mean()
avg_mae  = comparison_df[['MAE_M1', 'MAE_M2', 'MAE_M3']].mean()

# --- Create clean summary table ---
overall_best_df = pd.DataFrame({
    'Metric': ['Average_RMSE', 'Average_MAE'],
    'M1': [avg_rmse['RMSE_M1'], avg_mae['MAE_M1']],
    'M2': [avg_rmse['RMSE_M2'], avg_mae['MAE_M2']],
    'M3': [avg_rmse['RMSE_M3'], avg_mae['MAE_M3']]
})

# --- Add Best Model column ---
overall_best_df['Best_Model'] = overall_best_df[['M1', 'M2', 'M3']].idxmin(axis=1)

# --- Round values ---
overall_best_df[['M1','M2','M3']] = overall_best_df[['M1','M2','M3']].round(3)

# --- Styled table display ---
styled_table = overall_best_df.style.set_properties(**{
    'text-align': 'center'
}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'center')]}
])

display(styled_table)

# --- Save to CSV ---
overall_best_path = os.path.join(COMPARE_DIR, "lstm_overall_best_model.csv")
overall_best_df.to_csv(overall_best_path, index=False)

print(f"Overall best model summary saved at:\n{overall_best_path}")

,Metric,M1,M2,M3,Best_Model
0,Average_RMSE,229.858000,192.793000,206.830000,M2
1,Average_MAE,181.478000,151.203000,168.429000,M2


Overall best model summary saved at:
/content/drive/MyDrive/major_data/Compared_LSTM/lstm_overall_best_model.csv


In [ ]:
# Compare LSTM2(LSTM best model) vs GRU independent

In [7]:
import pandas as pd
from IPython.display import display

# --- File paths ---
LSTM2_FILE = "/content/drive/MyDrive/major_data/LSTM_2/lstm2_metrics/A_lstm2_test_metrics.csv"
GRU_FILE   = "/content/drive/MyDrive/major_data/GRU_INDEPENDENT/gru_ind_metrics/A_gru_ind_test_metrics.csv"


# --- Load CSVs ---
df_lstm2 = pd.read_csv(LSTM2_FILE)
df_gru   = pd.read_csv(GRU_FILE)


# --- Standardize Company column (if needed) ---
if 'Company' not in df_lstm2.columns:
    raise ValueError("LSTM CSV must contain 'Company' column")

if 'Company' not in df_gru.columns:
    raise ValueError("GRU CSV must contain 'Company' column")


# --- Rename columns for clarity ---
df_lstm2 = df_lstm2.rename(columns={
    'RMSE': 'RMSE_LSTM2',
    'MAE':  'MAE_LSTM2'
})

df_gru = df_gru.rename(columns={
    'RMSE': 'RMSE_GRU_IND',
    'MAE':  'MAE_GRU_IND'
})


# --- Merge both models ---
comparison_df = pd.merge(df_lstm2, df_gru, on='Company', how='inner')


# --- Sort by best model (lower RMSE is better) ---
comparison_df = comparison_df.sort_values(by='RMSE_LSTM2').reset_index(drop=True)


# --- Clean display formatting ---
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# --- Show table ---
print(" LSTM Model2 vs GRU Independent Comparison\n")
display(comparison_df)

 LSTM Model2 vs GRU Independent Comparison



,Company,RMSE_LSTM2,MAE_LSTM2,RMSE_GRU_IND,MAE_GRU_IND
0,Oil and Natural Gas Corporation Ltd,9.642560,7.462361,5.787562,4.200545
1,Bharat Electronics Ltd,12.617138,10.336692,22.701915,20.621768
2,Larsen and Toubro Ltd,12.721154,9.755460,39.432019,31.795770
3,JSW Steel Ltd,12.839587,9.783097,35.113863,30.744042
4,Adani ports and special Economic,15.164246,12.291716,30.461514,26.388150
5,Power Grid Corporation of India Ltd,18.565092,7.083663,17.169605,7.227202
6,ITC Ltd,20.357940,15.444626,37.665544,29.958003
7,Infosys Ltd,31.927222,23.784434,33.682603,24.868232
8,Eternal Ltd,32.375934,24.710750,34.656012,28.645582
9,Hindustan Unilever Ltd,43.665701,33.199733,38.982956,29.378419


In [8]:
# Add a column showing which model has the lowest RMSE
comparison_df['Best_Model_RMSE'] = comparison_df[['RMSE_LSTM2', 'RMSE_GRU_IND']].idxmin(axis=1)

# Clean label (remove RMSE_)
comparison_df['Best_Model_RMSE'] = comparison_df['Best_Model_RMSE'].str.replace('RMSE_', '')

# Display clean result table
result_df = comparison_df[['Company', 'RMSE_LSTM2', 'RMSE_GRU_IND', 'Best_Model_RMSE']]

print(result_df)

                                Company  RMSE_LSTM2  RMSE_GRU_IND Best_Model_RMSE
0   Oil and Natural Gas Corporation Ltd    9.642560      5.787562         GRU_IND
1                Bharat Electronics Ltd   12.617138     22.701915           LSTM2
2                 Larsen and Toubro Ltd   12.721154     39.432019           LSTM2
3                         JSW Steel Ltd   12.839587     35.113863           LSTM2
4      Adani ports and special Economic   15.164246     30.461514           LSTM2
5   Power Grid Corporation of India Ltd   18.565092     17.169605         GRU_IND
6                               ITC Ltd   20.357940     37.665544           LSTM2
7                           Infosys Ltd   31.927222     33.682603           LSTM2
8                           Eternal Ltd   32.375934     34.656012           LSTM2
9                Hindustan Unilever Ltd   43.665701     38.982956         GRU_IND
10                        Axis Bank Ltd   64.212781    178.036976           LSTM2
11              

In [9]:
import os

# --- Folder to save comparison CSVs ---
COMPARE_DIR = "/content/drive/MyDrive/major_data/Compare_LSTM_GRU"
os.makedirs(COMPARE_DIR, exist_ok=True)

# --- Save full comparison table ---
full_comparison_path = os.path.join(COMPARE_DIR, "lstm2_vs_gru_independent_full_comparison.csv")
comparison_df.to_csv(full_comparison_path, index=False)

print(f"Full comparison saved at: {full_comparison_path}")


# --- Save table with best RMSE per company ---
best_rmse_df = comparison_df[['Company', 'RMSE_LSTM2', 'RMSE_GRU_IND', 'Best_Model_RMSE']]

best_comparison_path = os.path.join(COMPARE_DIR, "lstm2_vs_gru_independent_best_model_rmse.csv")
best_rmse_df.to_csv(best_comparison_path, index=False)

print(f"Best RMSE per company saved at: {best_comparison_path}")

Full comparison saved at: /content/drive/MyDrive/major_data/Compare_LSTM_GRU/lstm2_vs_gru_independent_full_comparison.csv
Best RMSE per company saved at: /content/drive/MyDrive/major_data/Compare_LSTM_GRU/lstm2_vs_gru_independent_best_model_rmse.csv


In [10]:
import os
import pandas as pd
from IPython.display import display

# --- Calculate average RMSE and MAE per model ---
avg_rmse = comparison_df[['RMSE_LSTM2', 'RMSE_GRU_IND']].mean()
avg_mae  = comparison_df[['MAE_LSTM2', 'MAE_GRU_IND']].mean()

# --- Create clean summary table ---
summary_df = pd.DataFrame({
    'Metric': ['Average_RMSE', 'Average_MAE'],
    'LSTM2': [
        avg_rmse['RMSE_LSTM2'],
        avg_mae['MAE_LSTM2']
    ],
    'GRU_IND': [
        avg_rmse['RMSE_GRU_IND'],
        avg_mae['MAE_GRU_IND']
    ]
})

# --- Determine best model per metric ---
summary_df['Best_Model'] = summary_df[['LSTM2', 'GRU_IND']].idxmin(axis=1)

# --- Display table ---
display(summary_df)


# ================= SAVE TO DRIVE =================
COMPARE_DIR = "/content/drive/MyDrive/major_data/Compare_LSTM_GRU"
os.makedirs(COMPARE_DIR, exist_ok=True)

output_path = os.path.join(
    COMPARE_DIR,
    "lstm2_vs_gru_independent_overall_best_model.csv"
)

summary_df.to_csv(output_path, index=False)

print(f"\nOverall best model summary saved at:\n{output_path}")

,Metric,LSTM2,GRU_IND,Best_Model
0,Average_RMSE,192.792788,374.432455,LSTM2
1,Average_MAE,151.202960,329.712068,LSTM2



Overall best model summary saved at:
/content/drive/MyDrive/major_data/Compare_LSTM_GRU/lstm2_vs_gru_independent_overall_best_model.csv


In [ ]:
# COMPARISON OF LSTM2(LSTM best model) and GRU DEPENDENT detailed code is also given in 07_gru_dependent.ipynb

In [12]:
import pandas as pd
import os
from IPython.display import display

# -----------------------------
# Load saved metrics (IMPORTANT)
# -----------------------------
LSTM2_PATH = "/content/drive/MyDrive/major_data/LSTM_2/lstm2_metrics/A_lstm2_test_metrics.csv"
GRU_DEP_PATH = "/content/drive/MyDrive/major_data/GRU_DEPENDENT/gru_dep_metrics/A_gru_dep_test_metrics.csv"

metrics_df = pd.read_csv(LSTM2_PATH)
gru_dep_metrics_df = pd.read_csv(GRU_DEP_PATH)

# -----------------------------
# Calculate averages (from saved data)
# -----------------------------
avg_rmse_lstm2 = metrics_df['RMSE'].mean()
avg_mae_lstm2  = metrics_df['MAE'].mean()

avg_rmse_gru_dep = gru_dep_metrics_df['RMSE'].mean()
avg_mae_gru_dep  = gru_dep_metrics_df['MAE'].mean()

# -----------------------------
# Create summary table
# -----------------------------
summary_df = pd.DataFrame({
    'Metric': ['Average_RMSE', 'Average_MAE'],
    'LSTM2': [avg_rmse_lstm2, avg_mae_lstm2],
    'GRU_DEP': [avg_rmse_gru_dep, avg_mae_gru_dep]
})

# -----------------------------
# Determine best model
# -----------------------------
summary_df['Best_Model'] = summary_df[['LSTM2', 'GRU_DEP']].idxmin(axis=1)

# -----------------------------
# Display
# -----------------------------
print("\n LSTM2 vs GRU Dependent - Overall Performance\n")
display(summary_df)

# -----------------------------
# Save to CSV
# -----------------------------
COMPARE_DIR = "/content/drive/MyDrive/major_data/Compare_LSTM_GRU"
os.makedirs(COMPARE_DIR, exist_ok=True)

file_path = os.path.join(COMPARE_DIR, "lstm2_vs_gru_dep_overall_summary.csv")
summary_df.to_csv(file_path, index=False)

print(f"\n Summary saved at: {file_path}")


 LSTM2 vs GRU Dependent - Overall Performance



,Metric,LSTM2,GRU_DEP,Best_Model
0,Average_RMSE,192.792788,126.177628,GRU_DEP
1,Average_MAE,151.202960,91.330300,GRU_DEP



 Summary saved at: /content/drive/MyDrive/major_data/Compare_LSTM_GRU/lstm2_vs_gru_dep_overall_summary.csv


In [ ]:
# COMPARISON OF ALL MODELS

In [11]:
import pandas as pd
import os
from IPython.display import display

# -----------------------------
# Load ALL saved metrics
# -----------------------------
LSTM1_PATH = "/content/drive/MyDrive/major_data/LSTM_1/lstm1_metrics/A_lstm1_test_metrics.csv"
LSTM2_PATH = "/content/drive/MyDrive/major_data/LSTM_2/lstm2_metrics/A_lstm2_test_metrics.csv"
LSTM3_PATH = "/content/drive/MyDrive/major_data/LSTM_3/lstm3_metrics/A_lstm3_test_metrics.csv"

GRU_DEP_PATH = "/content/drive/MyDrive/major_data/GRU_DEPENDENT/gru_dep_metrics/A_gru_dep_test_metrics.csv"
GRU_IND_PATH = "/content/drive/MyDrive/major_data/GRU_INDEPENDENT/gru_ind_metrics/A_gru_ind_test_metrics.csv"

lstm1_df = pd.read_csv(LSTM1_PATH)
lstm2_df = pd.read_csv(LSTM2_PATH)
lstm3_df = pd.read_csv(LSTM3_PATH)
gru_dep_df = pd.read_csv(GRU_DEP_PATH)
gru_ind_df = pd.read_csv(GRU_IND_PATH)

# -----------------------------
# Calculate averages
# -----------------------------
avg_values = {
    'LSTM_1': [lstm1_df['RMSE'].mean(), lstm1_df['MAE'].mean()],
    'LSTM_2': [lstm2_df['RMSE'].mean(), lstm2_df['MAE'].mean()],
    'LSTM_3': [lstm3_df['RMSE'].mean(), lstm3_df['MAE'].mean()],
    'GRU_DEP': [gru_dep_df['RMSE'].mean(), gru_dep_df['MAE'].mean()],
    'GRU_IND': [gru_ind_df['RMSE'].mean(), gru_ind_df['MAE'].mean()]
}

# -----------------------------
# Create table (YOUR REQUIRED FORMAT)
# -----------------------------
summary_table = pd.DataFrame(
    avg_values,
    index=['Avg_RMSE', 'Avg_MAE']
)

# -----------------------------
# Add BEST MODEL column
# -----------------------------
summary_table['BEST_MODEL'] = summary_table.idxmin(axis=1)

# -----------------------------
# Display table
# -----------------------------
print("\n ALL MODELS COMPARISON TABLE (FINAL FORMAT)\n")
display(summary_table)

# -----------------------------
# Final Best Model (RMSE priority)
# -----------------------------
final_best_model = summary_table.loc['Avg_RMSE', 'BEST_MODEL']

print(f"\n FINAL BEST MODEL: {final_best_model}")

# -----------------------------
# Save CSV
# -----------------------------
COMPARE_DIR = "/content/drive/MyDrive/major_data/Compared_All_Models"
os.makedirs(COMPARE_DIR, exist_ok=True)

file_path = os.path.join(COMPARE_DIR, "all_5_models_matrix_comparison.csv")
summary_table.to_csv(file_path)

print(f"\n Saved at: {file_path}")


 ALL MODELS COMPARISON TABLE (FINAL FORMAT)



,LSTM_1,LSTM_2,LSTM_3,GRU_DEP,GRU_IND,BEST_MODEL
Avg_RMSE,229.858328,192.792788,206.829753,126.177628,374.432455,GRU_DEP
Avg_MAE,181.478046,151.202960,168.428927,91.330300,329.712068,GRU_DEP



 FINAL BEST MODEL: GRU_DEP

 Saved at: /content/drive/MyDrive/major_data/Compared_All_Models/all_5_models_matrix_comparison.csv
